# Generate factuality questions

In this notebook we query an LLM with the annotated dataset to generate

- Question to SmolDOC document
- Answer to question

and save it to disk, such that we do not have to ask the LLM multiple times (saving time and costs).

In [12]:
from utils import get_extended_datasets

datasets = get_extended_datasets()

example_cfg = 'smoldoc__en_sw' # We use Swahili since this config has the full 584 document translations
annotated_dataset = datasets[example_cfg]
annotated_dataset

📂 Found existing SmolDoc DatasetDict at data/smoldoc_datasets, loading from disk...
📂 Loaded DatasetDict from data/smoldoc_datasets with 102 configs.
Using cached file: data/smoldoc-factuality-ratings.json


Dataset({
    features: ['id', 'sl', 'tl', 'srcs', 'trgs', 'factuality', 'is_src_orig', 'annotator_1_label', 'annotator_1_notes', 'annotator_2_label', 'annotator_2_notes', 'annotator_3_label', 'annotator_3_notes'],
    num_rows: 584
})

In [13]:
import pandas as pd

# Get only the incorrect entries
df = pd.DataFrame(annotated_dataset)
incorrect_data = df[df["factuality"] == "has_errors"][:5] # Limit to first 5 examples
incorrect_data

,id,sl,tl,srcs,trgs,factuality,is_src_orig,annotator_1_label,annotator_1_notes,annotator_2_label,annotator_2_notes,annotator_3_label,annotator_3_notes
4,ethiopia_challenges__btithihhtt,en,sw,"[But in 1974, a military junta known as the De...","[Lakini mnamo 1974, kikosi cha wanamgambo kili...",has_errors,True,Minor Issue(s),"This paragraph contains minor issue: the ""peac...",No Issues,This is a correct account of Ethiopia's histro...,No Issues,The claims made in the Paragraph about the Eth...
6,topic_260__mtaftfttit,en,sw,[Movies have long been a powerful force in sha...,[Filamu zimekuwa na ushawishi mkubwa mno katik...,has_errors,True,Minor Issue(s),"This paragraph is mostly accurate, but there's...",Minor Issue(s),The claim regarding 12% of female characters w...,Not Sure,The claims are generally true. But I could not...
13,topic_493__isitgsgshgsg,en,sw,[Indira Gandhi was the first and only woman to...,[Indira Gandhi alikuwa mwanamke wa kwanza na w...,has_errors,True,Minor Issue(s),There are minor inaccuracies in this paragraph...,Minor Issue(s),Indira became member of the Parliament in 1964...,Minor Issue(s),Most of the biography about Indira Gandhi is a...
23,topic_131__gtttgtiigt,en,sw,[Grace: Can you tell me a little bit about its...,[Grace: Unaweza kunielezea kidogo kuhusu histo...,has_errors,True,Minor Issue(s),Tea was actually introduced in North America i...,Minor Issue(s),Tea was actually introduced in North America i...,Minor Issue(s),The history of tea in China is accurate. Howev...
33,custom_4__iifdfdfd,en,sw,"[I'm Dr. Boakye, a pediatrician at the Korle B...","[Mimi ni Dkt. Boakye, daktari wa watoto katika...",has_errors,True,Minor Issue(s),The cases reported are more than the real ones...,Clear Issue(s),There were approximately 3.5 million reported ...,Clear Issue(s),"The claim that in 2020, there were over 24 mil..."


In [23]:
QUESTION_GEN_SYSTEM_PROMPT = """
You are an expert dataset creator for factuality evaluation.
Your task is to read an English document (from the SmolDoc dataset) and produce a small set of factual question, answer pairs that test a model's factual understanding of the text.

Your goals:
1. Create one or more (up to three) concise, factual, and self-contained questions based on the given document.
2. Each question must have one short, unambiguous gold answer that is explicitly supported by the text.
3. Questions should be neither trivial nor adversarial — they should test meaningful factual comprehension, not obscure details or wordplay.

Output only valid JSON, following this structure:

[
  {
    "question": "<English question>",
    "answer": "<short correct English answer>"
  }
]

- Limit each question to less than 25 words.
- Limit each answer to less than 10 words.
"""

In [25]:
# id, question, answer

from llm_chat import CachedLLMChat, LLMChat, OllamaChatter, AzureOpenAIChatter

chatter = OllamaChatter(model_name="deepseek-r1:8b", think=False)
# chatter = AzureOpenAIChatter()
chat = LLMChat(chatter)
chat = CachedLLMChat(chat, cache_file_path="data/factuality_question_gen_cache.pkl")

questions_with_answers = []


def parse_response(response: str, id: str):
    """
    Parse the LLM response as JSON and attach topic_id.

    Args:
        response (str): The LLM response string.
        id (str): The topic ID to attach.
    Returns:
        list[dict] | None: The parsed JSON with topic_id added, or None on failure.
    """
    try:
        import json
        json_data = response
        loaded_json = json.loads(json_data)
        for entry in loaded_json:
            entry["topic_id"] = id
        return loaded_json
    except Exception as e:
        print(f"Error parsing response for id {id}: {e}")
        print(f"Response was: {response}")
        return None


for idx, row in incorrect_data.iterrows():
    id = row["id"]
    srcs = " ".join(row["srcs"])
    errors = "\n\n".join((row["annotator_1_notes"], row["annotator_2_notes"], row["annotator_3_notes"]))
    chat.add_message("system", QUESTION_GEN_SYSTEM_PROMPT)
    response, thoughts = chat.chat(
        f"""Here is the source document: {srcs}\n
        Annotators have noted the following issues:
        {errors}\n
        Generate question and answers in JSON format."""
    )
    parsed_json = parse_response(response, id)
    print(parsed_json)
    break



[{'question': 'Which group overthrew the Derg in 1991?', 'answer': "Ethiopian People's Revolutionary Democratic Front", 'topic_id': 'ethiopia_challenges__btithihhtt'}, {'question': 'What did Prime Minister Abiy Ahmed pledge to do in 2018?', 'answer': 'Reform the country', 'topic_id': 'ethiopia_challenges__btithihhtt'}, {'question': 'What group did Abiy Ahmed begin peace talks with in 2018?', 'answer': "Tigray People's Liberation Front", 'topic_id': 'ethiopia_challenges__btithihhtt'}]
